#### **Step 1: Import Required Libraries**

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import os

StatementMeta(, de7ded84-07c5-4f08-b3c4-c09608f1068c, 4, Finished, Available, Finished)

#### **Step 2: Overview: dim_customers**
This PySpark script builds a Slowly Changing Dimension Type 2 (SCD Type 2) dim_customers table for a Customer Data Platform (CDP) in a sports betting environment.
It reads cleaned silver-layer data, builds enriched customer profiles, tracks historical changes, and writes updates to a Delta Lake gold-layer dimension table, ensuring historical records are preserved.

Key Features
- Joins cleaned data from users, preferences, and devices
- Aggregates device stats
- Tracks changes over time with SCD Type 2
- Deduplicates records before writing
- Supports schema auto-merge

In [2]:
# Initialize SparkSession
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# === Define target Delta table path (gold layer) ===
gold_path = "Files/cdp_gold/dim_customers"

# === Load silver-layer source tables ===
df_users = spark.read.parquet("Files/cdp_silver/oracle/cleaned/cleaned_users")
df_preferences = spark.read.parquet("Files/cdp_silver/oracle/cleaned/cleaned_user_preferences")
df_devices = spark.read.parquet("Files/cdp_silver/postgresql/cleaned/cleaned_user_devices")

# === Step 1: Prepare source DataFrames ===

# User profile data
users_df = df_users.select(
    "user_id", "username", "email", "first_name", "last_name",
    "date_of_birth", "registration_date", "country_code",
    "kyc_status", "account_status", "load_timestamp"
)

# User preference data
preferences_df = df_preferences.select(
    "user_id", "favorite_sports", "odds_format", "marketing_opt_in", "load_timestamp"
)

# Device data: get most recent device per user
device_window = Window.partitionBy("user_id").orderBy(col("load_timestamp").desc())

devices_df = df_devices.withColumn("row_num", row_number().over(device_window)) \
    .filter(col("row_num") == 1) \
    .select("user_id", "device_type", "ip_address")

# Aggregate device counts
device_count_df = df_devices.groupBy("user_id").agg(
    countDistinct("device_id").alias("device_count")
)

# === Step 2: Assemble enriched customer profile ===
current_df = users_df \
    .join(preferences_df, on="user_id", how="left") \
    .join(devices_df, on="user_id", how="left") \
    .join(device_count_df, on="user_id", how="left") \
    .withColumn("effective_date", current_timestamp()) \
    .withColumn("end_date", lit(None).cast("timestamp")) \
    .withColumn("is_current", lit(True)) \
    .withColumn("customer_key", row_number().over(Window.orderBy("user_id"))) \
    .select(
        "customer_key", "user_id", "username", "email", "first_name", "last_name",
        "date_of_birth", "registration_date", "country_code",
        "kyc_status", "account_status", "favorite_sports", "odds_format",
        col("marketing_opt_in").cast("boolean"),
        "device_count", col("device_type").alias("last_device_type"),
        col("ip_address").alias("last_ip_address"),
        "effective_date", "end_date", "is_current"
    )

# === Step 3: Deduplicate customer profiles ===
dedup_columns = [
    "user_id", "username", "email", "first_name", "last_name", "date_of_birth",
    "registration_date", "country_code", "kyc_status", "account_status",
    "favorite_sports", "odds_format", "marketing_opt_in",
    "device_count", "last_device_type", "last_ip_address"
]
current_df = current_df.dropDuplicates(dedup_columns)

# === Step 4: Perform SCD Type 2 logic ===
if os.path.exists(gold_path):
    # Load existing gold table
    gold_table = DeltaTable.forPath(spark, gold_path)
    existing_df = gold_table.toDF().filter("is_current = True")

    # Detect changes between existing and incoming data
    change_condition = " OR ".join([f"source.{col} <> target.{col}" for col in dedup_columns])

    staged_changes = current_df.alias("source").join(
        existing_df.alias("target"),
        on="user_id", how="left"
    ).filter(change_condition)

    # === 4a: Mark current record as inactive (close out old version) ===
    updates_df = staged_changes.select("user_id").distinct().withColumn("end_date", current_timestamp())
    gold_table.alias("target").merge(
        updates_df.alias("updates"),
        "target.user_id = updates.user_id AND target.is_current = True"
    ).whenMatchedUpdate(set={
        "is_current": lit(False),
        "end_date": updates_df["end_date"]
    }).execute()

    # === 4b: Insert new versioned records ===
    staged_changes = staged_changes.drop("customer_key") \
        .withColumn("customer_key", row_number().over(Window.orderBy("user_id")))

    gold_table.alias("target").merge(
        staged_changes.alias("source"),
        "target.user_id = source.user_id AND target.is_current = True"
    ).whenNotMatchedInsertAll().execute()

else:
    # First-time load
    current_df.write.format("delta").mode("overwrite").save(gold_path)

StatementMeta(, 9100725b-5fb8-4887-bbb8-b91fe2c90c87, 4, Finished, Available, Finished)

#### **Step 3: Overview: fact_user_transactions**
Goal: Track user transaction behavior from the cleaned_transactions table in the silver layer.

Key Features:
- Incremental load (based on load_timestamp)
- Deduplication to prevent duplicated metrics
- Aggregation of monetary KPIs per user
- Delta merge logic for upserts

In [5]:
# === Spark Session with Delta merge enabled ===
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# === Paths ===
silver_txn_path = "Files/cdp_silver/postgresql/cleaned/cleaned_transactions"
gold_fact_path = "Files/cdp_gold/fact_user_transactions"

# === Load Silver Transactions ===
df_txns = spark.read.parquet(silver_txn_path)

# === Load max load_timestamp from gold (for incremental load) ===
if os.path.exists(gold_fact_path):
    gold_table = DeltaTable.forPath(spark, gold_fact_path)
    latest_ts = gold_table.toDF().agg({"transaction_load_timestamp": "max"}).collect()[0][0]
else:
    latest_ts = None

# === Filter silver data incrementally ===
if latest_ts:
    df_txns = df_txns.filter(col("load_timestamp") > lit(latest_ts))

# === Deduplicate based on txn_id and load_timestamp ===
df_txns = df_txns.dropDuplicates(["txn_id", "load_timestamp"])

# === Compute Aggregations per user ===
agg_df = df_txns.groupBy("user_id").agg(
    round(sum(when(col("txn_type") == "Deposit", col("amount")).otherwise(0)), 2).alias("total_deposits"),
    round(sum(when(col("txn_type") == "Withdrawal", col("amount")).otherwise(0)), 2).alias("total_withdrawals"),
    count("*").alias("number_of_transactions"),
    round(avg("amount"), 2).alias("average_transaction"),
    max("load_timestamp").alias("last_transaction_time")
)

# Use payment method mode() manually (most frequent method)
preferred_method_df = df_txns.groupBy("user_id", "payment_method") \
    .agg(count("*").alias("method_count"))

window_spec = Window.partitionBy("user_id").orderBy(col("method_count").desc())

preferred_method_df = preferred_method_df.withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") == 1) \
    .select("user_id", col("payment_method").alias("preferred_payment_method"))

# Join it back
final_df = agg_df.join(preferred_method_df, on="user_id", how="left") \
    .withColumn("transaction_load_timestamp", current_timestamp()) \
    .withColumn("transaction_key", row_number().over(Window.orderBy("user_id"))) \
    .select(
        "transaction_key", "user_id", "total_deposits", "total_withdrawals",
        "number_of_transactions", "average_transaction", "last_transaction_time",
        "preferred_payment_method", "transaction_load_timestamp"
    )

# === Upsert into gold.fact_user_transactions ===
if os.path.exists(gold_fact_path):
    gold_table = DeltaTable.forPath(spark, gold_fact_path)

    gold_table.alias("target").merge(
        final_df.alias("source"),
        "target.user_id = source.user_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

else:
    final_df.write.format("delta").mode("overwrite").save(gold_fact_path)


StatementMeta(, de7ded84-07c5-4f08-b3c4-c09608f1068c, 7, Finished, Available, Finished)

#### **Step 3: Overview: fact_betting_summary**
Purpose: Track user transaction behavior from the cleaned_transactions table in the silver layer.

Key Outputs:
- Load Type: Incremental
- Duplicates:  Removed
- Total bets, win rate, average odds/stakes
- Favorite league and channel per user
- Time-aware gold-layer table using Delta Lake

In [4]:
# === Initialize Spark session ===
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# === Define paths ===
bet_events_path = "Files/cdp_silver/kafka_evenstream/cleaned/cleaned_betting_events"
bet_results_path = "Files/cdp_silver/kafka_evenstream/cleaned/cleaned_bet_results"
match_metadata_path = "Files/cdp_silver/kafka_evenstream/cleaned/cleaned_match_metdata"
gold_path = "Files/cdp_gold/fact_betting_summary"

# === Load source tables ===
df_bets = spark.read.parquet(bet_events_path)
df_results = spark.read.parquet(bet_results_path)
df_matches = spark.read.parquet(match_metadata_path)

# === Get max load timestamp for incremental loading ===
if os.path.exists(gold_path):
    gold_table = DeltaTable.forPath(spark, gold_path)
    latest_ts = gold_table.toDF().agg({"betting_load_timestamp": "max"}).collect()[0][0]
else:
    latest_ts = None

# === Incremental filter ===
if latest_ts:
    df_bets = df_bets.filter(col("bet_time") > lit(latest_ts))
    df_results = df_results.filter(col("settlement_time") > lit(latest_ts))

# === Deduplicate ===
df_bets = df_bets.dropDuplicates(["event_id", "user_id", "match_id"])
df_results = df_results.dropDuplicates(["event_id", "user_id", "match_id"])

# === Join bets and results ===
bet_with_result_df = df_bets.alias("b").join(
    df_results.alias("r"),
    on=["event_id", "user_id", "match_id"],
    how="left"
)

# === Join match metadata for league info ===
bet_with_meta_df = bet_with_result_df.join(
    df_matches.select("match_id", "league"),
    on="match_id", how="left"
)

# === Aggregate metrics per user ===
agg_df = bet_with_meta_df.groupBy("user_id").agg(
    count("*").alias("total_bets"),
    sum("stake_amount").alias("total_staked"),
    round(avg("stake_amount"), 2).alias("average_stake"),
    round(avg("odds"), 2).alias("average_odds"),
    expr("sum(case when outcome = 'Win' then 1 else 0 end) / count(*)").alias("win_rate"),
    max("bet_time").alias("last_bet_time")
)

# === Favorite league (most frequent per user) ===
favorite_league_df = bet_with_meta_df.groupBy("user_id", "league") \
    .agg(count("*").alias("league_count"))

league_window = Window.partitionBy("user_id").orderBy(col("league_count").desc())
favorite_league_df = favorite_league_df.withColumn("rank", row_number().over(league_window)) \
    .filter(col("rank") == 1) \
    .select("user_id", col("league").alias("favorite_league"))

# === Preferred channel (Web/Mobile) ===
preferred_channel_df = df_bets.groupBy("user_id", "channel") \
    .agg(count("*").alias("channel_count"))

channel_window = Window.partitionBy("user_id").orderBy(col("channel_count").desc())
preferred_channel_df = preferred_channel_df.withColumn("rank", row_number().over(channel_window)) \
    .filter(col("rank") == 1) \
    .select("user_id", col("channel").alias("preferred_channel"))

# === Registration dates (from user dim) for conversion rate ===
dim_users_path = "Files/cdp_gold/dim_customers"
df_users = spark.read.format("delta").load(dim_users_path).filter("is_current = true")
df_users = df_users.select("user_id", "registration_date")

# === Combine all metrics ===
final_df = agg_df \
    .join(favorite_league_df, on="user_id", how="left") \
    .join(preferred_channel_df, on="user_id", how="left") \
    .join(df_users, on="user_id", how="left") \
    .withColumn("bet_conversion_rate",
                round(col("total_bets") / datediff(current_date(), to_date("registration_date")).cast("double"), 2)) \
    .withColumn("betting_load_timestamp", current_timestamp()) \
    .withColumn("betting_summary_key", row_number().over(Window.orderBy("user_id"))) \
    .select(
        "betting_summary_key", "user_id", "total_bets", "total_staked", "average_stake",
        "average_odds", "win_rate", "bet_conversion_rate",
        "favorite_league", "preferred_channel", "betting_load_timestamp"
    )

# === Merge into gold.fact_betting_summary table ===
if os.path.exists(gold_path):
    gold_table = DeltaTable.forPath(spark, gold_path)

    gold_table.alias("target").merge(
        final_df.alias("source"),
        "target.user_id = source.user_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

else:
    final_df.write.format("delta").mode("overwrite").save(gold_path)


StatementMeta(, 9100725b-5fb8-4887-bbb8-b91fe2c90c87, 6, Finished, Available, Finished)

#### **Step 4: Overview – fact_match_analytics**
Purpose:
Aggregates betting activity, payout totals, and odds volatility per match.

Tables Used:

- cleaned_betting_events
- cleaned_bet_results
- cleaned_live_odds_updates
- cleaned_match_metadata

Key Outputs:

- Load Type: Incremental
- Duplicates: Removed
- Total bets, total payout, average odds
- Odds volatility score
- Final match outcome
- Match timestamp & metadata from silver

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import os

# === Spark Session Setup ===
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# === Define Paths ===
bets_path = "Files/cdp_silver/kafka_evenstream/cleaned/cleaned_betting_events"
results_path = "Files/cdp_silver/kafka_evenstream/cleaned/cleaned_bet_results"
odds_path = "Files/cdp_silver/kafka_evenstream/cleaned/cleaned_live_odds_update"
matches_path = "Files/cdp_silver/kafka_evenstream/cleaned/cleaned_match_metdata"
gold_path = "Files/cdp_gold/fact_match_analytics"

# === Load silver data ===
df_bets = spark.read.parquet(bets_path)
df_results = spark.read.parquet(results_path)
df_odds = spark.read.parquet(odds_path)
df_matches = spark.read.parquet(matches_path)

# === Incremental Filtering ===
if os.path.exists(gold_path):
    gold_table = DeltaTable.forPath(spark, gold_path)
    latest_ts = gold_table.toDF().agg({"match_load_timestamp": "max"}).collect()[0][0]
else:
    latest_ts = None

if latest_ts:
    df_bets = df_bets.filter(col("bet_time") > lit(latest_ts))
    df_results = df_results.filter(col("settlement_time") > lit(latest_ts))
    df_odds = df_odds.filter(col("timestamp") > lit(latest_ts))
    df_matches = df_matches.filter(col("event_date") > lit(latest_ts))

# === Deduplicate Events ===
df_bets = df_bets.dropDuplicates(["event_id", "user_id", "match_id"])
df_results = df_results.dropDuplicates(["event_id", "user_id", "match_id"])
df_odds = df_odds.dropDuplicates(["update_id", "match_id", "timestamp"])

# === Match Metadata ===
matches_df = df_matches.select("match_id", "sport", "league", "event_date").dropDuplicates()

# === Join match metadata to other sources to preserve sport/league context ===
df_bets = df_bets.join(matches_df, on="match_id", how="left")
df_results = df_results.join(matches_df, on="match_id", how="left")
df_odds = df_odds.join(matches_df, on="match_id", how="left")

# === Aggregations by match_id, sport, league ===
bets_agg = df_bets.groupBy("match_id", "sport", "league").agg(
    count("*").alias("total_bets"),
    round(avg("odds"), 2).alias("average_odds")
)

payout_agg = df_results.groupBy("match_id", "sport", "league").agg(
    round(sum("settlement_amount"), 2).alias("total_payout")
)

# === Final Outcome (most frequent) per match/sport/league ===
outcome_freq_df = df_results.groupBy("match_id", "sport", "league", "outcome").agg(
    count("*").alias("outcome_count")
)
outcome_window = Window.partitionBy("match_id", "sport", "league").orderBy(col("outcome_count").desc())

final_outcome_df = outcome_freq_df.withColumn("rank", row_number().over(outcome_window)) \
    .filter(col("rank") == 1) \
    .select("match_id", "sport", "league", col("outcome").alias("final_outcome"))

# === Odds Volatility Score ===
volatility_df = df_odds.groupBy("match_id", "sport", "league").agg(
    stddev_samp("odds_home").alias("vol_home"),
    stddev_samp("odds_away").alias("vol_away")
).withColumn(
    "odds_volatility_score",
    round(coalesce(col("vol_home"), lit(0)) + coalesce(col("vol_away"), lit(0)), 2)
).select("match_id", "sport", "league", "odds_volatility_score")

# === Assemble Final Match Summary ===
final_df = matches_df \
    .join(bets_agg, on=["match_id", "sport", "league"], how="left") \
    .join(payout_agg, on=["match_id", "sport", "league"], how="left") \
    .join(final_outcome_df, on=["match_id", "sport", "league"], how="left") \
    .join(volatility_df, on=["match_id", "sport", "league"], how="left") \
    .withColumn("match_load_timestamp", current_timestamp()) \
    .withColumn("match_key", row_number().over(Window.orderBy("match_id", "sport", "league"))) \
    .select(
        "match_key", "match_id", "sport", "league","event_date",
        "total_bets", "total_payout", "average_odds",
        "odds_volatility_score", "final_outcome","match_load_timestamp"
    ) \
    .dropDuplicates(["match_id", "sport", "league"])

# === Upsert to Gold Table ===
if os.path.exists(gold_path):
    gold_table = DeltaTable.forPath(spark, gold_path)

    gold_table.alias("target").merge(
        final_df.alias("source"),
        "target.match_id = source.match_id AND target.sport = source.sport AND target.league = source.league"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
else:
    final_df.write.format("delta").mode("overwrite").save(gold_path)


StatementMeta(, 9100725b-5fb8-4887-bbb8-b91fe2c90c87, 7, Finished, Available, Finished)

#### **Step 5: Overview – fact_odds_movement**
Purpose:
- Tracks how betting odds for each market evolve throughout a match to assess volatility.
- 
Tables Used:
- cleaned_live_odds_update

Key Outputs:
- Load Type: Incremental
- Duplicates: Removed
- Start vs. end odds per market
- Odds volatility score (stddev)
- Last update timestamp

In [6]:
# === Spark Session Setup ===
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# === Paths ===
odds_path = "Files/cdp_silver/kafka_evenstream/cleaned/cleaned_live_odds_update"
gold_path = "Files/cdp_gold/fact_odds_movement"

# === Load cleaned odds data ===
df_odds = spark.read.parquet(odds_path)

# === Incremental load filter ===
if os.path.exists(gold_path):
    gold_table = DeltaTable.forPath(spark, gold_path)
    latest_ts = gold_table.toDF().agg({"last_update_time": "max"}).collect()[0][0]
    df_odds = df_odds.filter(col("timestamp") > lit(latest_ts))

# === Ensure relevant columns exist and are not null ===
df_odds = df_odds.filter(col("match_id").isNotNull() & col("market").isNotNull() & col("odds_home").isNotNull())

# === Get start and end odds per match and market ===
window_spec = Window.partitionBy("match_id", "market").orderBy("timestamp")

start_odds = df_odds.withColumn("start_odds_home", first("odds_home").over(window_spec))
end_odds = df_odds.withColumn("end_odds_home", last("odds_home").over(window_spec))

# === Compute volatility ===
volatility_df = df_odds.groupBy("match_id", "market").agg(
    stddev_samp("odds_home").alias("odds_volatility_score"),
    max("timestamp").alias("last_update_time")
)

# === Combine start, end, and volatility ===
final_df = start_odds \
    .join(end_odds.select("match_id", "market", "end_odds_home"), on=["match_id", "market"], how="inner") \
    .join(volatility_df, on=["match_id", "market"], how="inner") \
    .select(
        "match_id", "market",
        round("start_odds_home", 2).alias("start_odds_home"),
        round("end_odds_home", 2).alias("end_odds_home"),
        round("odds_volatility_score", 2).alias("odds_volatility_score"),
        date_format("last_update_time", "yyyy-MM-dd'T'HH:mm:ss").alias("last_update_time")
    ).dropDuplicates(["match_id", "market"])

# Add surrogate key
final_df = final_df.withColumn(
    "odds_movement_key", row_number().over(Window.orderBy("match_id", "market"))
).select(
    "odds_movement_key", "match_id", "market", "start_odds_home",
    "end_odds_home", "odds_volatility_score", "last_update_time"
)

# === Upsert to gold ===
if os.path.exists(gold_path):
    gold_table.alias("target").merge(
        final_df.alias("source"),
        "target.match_id = source.match_id AND target.market = source.market"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
else:
    final_df.write.format("delta").mode("overwrite").save(gold_path)


StatementMeta(, 9100725b-5fb8-4887-bbb8-b91fe2c90c87, 8, Finished, Available, Finished)

#### **Step 6: Overview – dim_user_segments**
Purpose:
- Classifies users into behavior-based segments for personalized targeting and marketing.

Tables Used:
- gold.fact_user_transactions
- gold.fact_betting_summary
- gold.dim_customers

Key Outputs:
- Load Type: Full/Refresh
- Segments by value, activity & behavior
- Engagement score & risk profile
- Last active date
- Time-aware gold-layer snapshot

In [7]:
# === Spark Session Setup ===
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# === Define Paths ===
bet_summary_path = "Files/cdp_gold/fact_betting_summary"
gold_path = "Files/cdp_gold/dim_user_segments"

# === Load source: Betting summary (from gold layer) ===
df_bets = spark.read.format("delta").load(bet_summary_path)

# === Incremental filtering ===
if os.path.exists(gold_path):
    gold_table = DeltaTable.forPath(spark, gold_path)
    latest_ts = gold_table.toDF().agg({"segment_load_timestamp": "max"}).collect()[0][0]
    df_bets = df_bets.filter(col("betting_load_timestamp") > lit(latest_ts))

# === Deduplicate based on user_id ===
df_bets = df_bets.dropDuplicates(["user_id"])

# === Define segmentation rules ===
df_segments = df_bets.withColumn(
    "segment_name", when(col("total_staked") > 10000, "High Value")
                    .when((col("total_staked") > 1000) & (col("total_bets") > 100), "Engaged")
                    .when(col("total_bets") < 10, "Inactive")
                    .otherwise("Moderate")
).withColumn(
    "risk_profile", when(col("average_odds") > 5, "High Risk")
                   .when(col("average_odds") > 2.5, "Medium Risk")
                   .otherwise("Low Risk")
).withColumn(
    "betting_style", when(col("average_stake") > 200, "Aggressive")
                    .when(col("average_stake") < 50, "Casual")
                    .otherwise("Moderate")
).withColumn(
    "engagement_score",
    round((col("total_bets") * col("win_rate") + col("total_staked") / 1000), 2)
).withColumn(
    "last_active_date", to_date(col("betting_load_timestamp"))
).withColumn(
    "segment_load_timestamp", current_timestamp()
).withColumn(
    "segment_key", row_number().over(Window.orderBy("user_id"))
).select(
    "segment_key", "user_id", "segment_name", "risk_profile", "betting_style",
    "engagement_score", "last_active_date", "segment_load_timestamp"
)

# === Upsert to gold layer ===
if os.path.exists(gold_path):
    gold_table.alias("target").merge(
        df_segments.alias("source"),
        "target.user_id = source.user_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
else:
    df_segments.write.format("delta").mode("overwrite").save(gold_path)


StatementMeta(, 9100725b-5fb8-4887-bbb8-b91fe2c90c87, 9, Finished, Available, Finished)